# Offline run analysis

Set `RUN_ID` to an existing run folder. This notebook loads that run's labeled training pass and realtime labeled trial, trains offline model variants, tests each on the realtime trial, and saves sweep outputs back into the same run folder.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'training.py').exists():
    ROOT = Path(r'D:/BME/BCI/online_bci/online_eeg')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from training import TrainingConfig, offline_train_test_sweep
from plots import plot_labeled_recording, plot_predictions_overlay

print('Pipeline root:', ROOT)


## Run and sweep settings

In [ ]:
RUN_ID = 'run_001'
RUN_DIR = ROOT / 'runs' / RUN_ID

TRAIN_LABELED_NPZ = RUN_DIR / 'labeled_training' / 'train_01_labeled.npz'
TEST_LABELED_NPZ = RUN_DIR / 'realtime_trials' / 'realtime_trial_01_raw_labeled.npz'
SWEEP_DIR = RUN_DIR / 'offline_sweeps'

FEATURE_MODES = ('raw_signal', 'fft_bandpower')
WINDOW_SECS = (0.75, 1.0, 1.5, 2.0)
STRIDE_SECS = (0.2,)
BANDPOWER_HZ_VALUES = ((8.0, 35.0), (8.0, 12.0), (13.0, 30.0), (18.0, 22.0))

TRAIN = TrainingConfig(
    train_fraction=0.80,
    hidden_size=32,
    num_layers=1,
    dropout=0.0,
    batch_size=64,
    epochs=20,
    lr=1e-3,
    seed=888,
)

if not RUN_DIR.exists():
    raise FileNotFoundError(f'Run folder not found: {RUN_DIR}')
if not TRAIN_LABELED_NPZ.exists():
    raise FileNotFoundError(f'Training labeled file not found: {TRAIN_LABELED_NPZ}')
if not TEST_LABELED_NPZ.exists():
    raise FileNotFoundError(f'Test labeled file not found: {TEST_LABELED_NPZ}')

SWEEP_DIR.mkdir(parents=True, exist_ok=True)
print('Run directory:', RUN_DIR)
print('Training file:', TRAIN_LABELED_NPZ)
print('Test file:', TEST_LABELED_NPZ)
print('Sweep output:', SWEEP_DIR)


## Inspect loaded train/test labels

In [ ]:
fig, ax = plot_labeled_recording(TRAIN_LABELED_NPZ, max_duration_sec=30.0)
ax.set_title('Training labeled EEG preview')


In [ ]:
fig, ax = plot_labeled_recording(TEST_LABELED_NPZ, max_duration_sec=None)
ax.set_title('Realtime trial labeled EEG preview')


## Run offline model sweep

In [ ]:
sweep_result = offline_train_test_sweep(
    train_labeled_npz=TRAIN_LABELED_NPZ,
    test_labeled_npz=TEST_LABELED_NPZ,
    output_dir=SWEEP_DIR,
    feature_modes=FEATURE_MODES,
    window_secs=WINDOW_SECS,
    stride_secs=STRIDE_SECS,
    bandpower_hz_values=BANDPOWER_HZ_VALUES,
    training_config=TRAIN,
)

summary = sweep_result['summary'].sort_values('test_balanced_accuracy', ascending=False).reset_index(drop=True)
print('Saved sweep summary:', sweep_result['summary_csv'])
display(summary)


## Compare variants

In [ ]:
plot_df = summary.copy()
plot_df['variant_short'] = plot_df['variant'].str.replace('__', '\n', regex=False)
ax = plot_df.plot.bar(
    x='variant_short',
    y=['val_balanced_accuracy', 'test_balanced_accuracy'],
    figsize=(14, 5),
)
ax.set_xlabel('Variant')
ax.set_ylabel('Balanced accuracy')
ax.set_title(f'Offline sweep results for {RUN_ID}')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()


## Inspect best variant predictions

In [ ]:
best = summary.iloc[0]
best_pred_csv = Path(best['variant_dir']) / f'{TEST_LABELED_NPZ.stem}_test_predictions.csv'
print('Best variant:', best['variant'])
print('Best predictions:', best_pred_csv)

best_predictions = pd.read_csv(best_pred_csv)
display(best_predictions.tail())
fig, ax = plot_predictions_overlay(TEST_LABELED_NPZ, best_predictions, max_duration_sec=None)
ax.set_title(f"Best offline variant on realtime trial: {best['variant']}")
